# **Durability — RUL DATASET GENERATION**

## Prerequisite

Run [`01_generate_dataset.ipynb`](01_generate_dataset.ipynb) $\to$
[`02_train_pce.ipynb`](02_train_pce.ipynb) $\to$
[`03_generate_dataset_nn.ipynb`](03_generate_dataset_nn.ipynb) $\to$
[`04_train_nn.ipynb`](04_train_nn.ipynb) first — this notebook loads the global NN
(`lambda 1`/`lambda 2` vs. `(fck, rh, cov, t)`) that stage 4 writes.

## What this notebook does

Fixes a single design point `(fck, rh, cov)` and sweeps a list of time steps through the trained
NN, giving $\lambda_1(t)$ and $\lambda_2(t)$ at each one **directly** — no need to pick a
per-time-step PCE first. $\lambda_3$ and $\lambda_4$ barely move across the design space, which is
why they were never modelled by the NN — they're fixed here instead, to a value you set or, by
default, the mean over the NN's own training dataset.

At each time step, `generate_rul_dataset_durability` builds a `GlamFKML(lam1, lam2, lam3, lam4)` and
draws `n_glam_samples` Monte Carlo realisations of the state limit function $g$ = cover −
carbonation depth — the raw material for the spaghetti plot and the RUL analysis in
[`05_plot_rul_analysis.ipynb`](05_plot_rul_analysis.ipynb).

## 1. Libraries

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import dill
import numpy as np
import pandas as pd

from functions import *

C:\git-projetos\2024-1_victor_hugo_renata_maria\.venv\Lib\site-packages\UQpy\__init__.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## 2. Config

`n_latent_samples`, `installation_year`, `co2_scenario`, `cement_type` and `exposure_conditions`
must match [`04_train_nn.ipynb`](04_train_nn.ipynb) — it names the NN model files being loaded.

`lambda3_fixed`/`lambda4_fixed` left as `None` fall back to the mean of `lambda 3`/`lambda 4` over
the NN training dataset; set either explicitly to override.

In [2]:
n_latent_samples    = 2500      # must match stage 4 — it names the NN model files being loaded
installation_year   = 1980
co2_scenario        = "SSP2-4.5"
cement_type         = 3
exposure_conditions = 2

fck_fixed = 25.0   # fixed compressive strength for the whole sweep (MPa)
rh_fixed  = 55.0   # fixed relative humidity for the whole sweep (%)
cov_fixed = 25.0   # fixed cover for the whole sweep (mm)
times     = np.linspace(0, 100, 20, endpoint=True)  # time steps to sweep through the NN — any grid, not just the training one

lambda3_fixed = 0.132011   # None = mean of lambda 3 over the NN training dataset
lambda4_fixed = 0.141281   # None = mean of lambda 4 over the NN training dataset

n_glam_samples = 50000   # Monte Carlo samples drawn from the GLD at each time step

print(f"Sweeping t = {times.min()}..{times.max()} years at fck={fck_fixed}, rh={rh_fixed}, cov={cov_fixed}")

Sweeping t = 0.0..100.0 years at fck=25.0, rh=55.0, cov=25.0


## 3. Predict lambda 1/2, fix lambda 3/4, and draw the GLD samples

In [3]:
print("="*60)
print("GENERATING THE RUL DATASET")
print("="*60)

result = generate_rul_dataset_durability(
                                            fck=fck_fixed,
                                            rh=rh_fixed,
                                            cov=cov_fixed,
                                            times=times,
                                            n_latent_samples=n_latent_samples,
                                            installation_year=installation_year,
                                            cement_type=cement_type,
                                            exposure_conditions=exposure_conditions,
                                            co2_scenario=co2_scenario,
                                            lambda3_fixed=lambda3_fixed,
                                            lambda4_fixed=lambda4_fixed,
                                            n_glam_samples=n_glam_samples,
                                            input_dir='.',
                                            output_dir='.',
                                         )

lambda_df = result['lambda_df']
samples   = result['samples']
print(f"\nSamples shape: {samples.shape}  (n_glam_samples x len(times))")
lambda_df

GENERATING THE RUL DATASET

----------------------------------------
GENERATING RUL DATASET AT fck=25.0, rh=55.0, cov=25.0
----------------------------------------
  lambda 3 fixed at 0.1320, lambda 4 fixed at 0.1413
  t = 0.0: lambda 1 = 25.574, lambda 2 = 2.932, sample mean = 25.572, sample std = 0.498
  t = 5.3: lambda 1 = 21.548, lambda 2 = 2.666, sample mean = 21.546, sample std = 0.548
  t = 10.5: lambda 1 = 17.202, lambda 2 = 2.389, sample mean = 17.200, sample std = 0.612
  t = 15.8: lambda 1 = 12.573, lambda 2 = 2.201, sample mean = 12.571, sample std = 0.664
  t = 21.1: lambda 1 = 8.072, lambda 2 = 1.945, sample mean = 8.070, sample std = 0.751
  t = 26.3: lambda 1 = 4.808, lambda 2 = 1.726, sample mean = 4.806, sample std = 0.847
  t = 31.6: lambda 1 = 2.750, lambda 2 = 1.620, sample mean = 2.748, sample std = 0.902
  t = 36.8: lambda 1 = 0.832, lambda 2 = 1.544, sample mean = 0.829, sample std = 0.947
  t = 42.1: lambda 1 = -1.047, lambda 2 = 1.482, sample mean = -1.050, sa

     fck    rh   cov  Time (years)   lambda 1  lambda 2  lambda 3  lambda 4
0   25.0  55.0  25.0      0.000000  25.573740  2.932099  0.132011  0.141281
1   25.0  55.0  25.0      5.263158  21.547876  2.666336  0.132011  0.141281
2   25.0  55.0  25.0     10.526316  17.201683  2.388709  0.132011  0.141281
3   25.0  55.0  25.0     15.789474  12.572575  2.201183  0.132011  0.141281
4   25.0  55.0  25.0     21.052632   8.071867  1.944751  0.132011  0.141281
5   25.0  55.0  25.0     26.315789   4.808018  1.726180  0.132011  0.141281
6   25.0  55.0  25.0     31.578947   2.750239  1.620250  0.132011  0.141281
7   25.0  55.0  25.0     36.842105   0.832228  1.543675  0.132011  0.141281
8   25.0  55.0  25.0     42.105263  -1.047274  1.482019  0.132011  0.141281
9   25.0  55.0  25.0     47.368421  -2.874354  1.418370  0.132011  0.141281
10  25.0  55.0  25.0     52.631579  -4.457366  1.354374  0.132011  0.141281
11  25.0  55.0  25.0     57.894737  -5.951472  1.303584  0.132011  0.141281
12  25.0  55

## 4. Sanity check

In [4]:
summary = pd.DataFrame({
                           'Time (years)': result['times'],
                           'Sample mean':  samples.mean(axis=0),
                           'Sample std':   samples.std(axis=0),
                           'P(g <= 0)':    (samples <= 0).mean(axis=0),
                        })
summary

    Time (years)  Sample mean  Sample std  P(g <= 0)
0       0.000000    25.572298    0.498422    0.00000
1       5.263158    21.546289    0.548102    0.00000
2      10.526316    17.199912    0.611805    0.00000
3      15.789474    12.570653    0.663926    0.00000
4      21.052632     8.069692    0.751471    0.00000
5      26.315789     4.805568    0.846623    0.00000
6      31.578947     2.747629    0.901974    0.00130
7      36.842105     0.829488    0.946717    0.18976
8      42.105263    -1.050128    0.986104    0.85868
9      47.368421    -2.877336    1.030354    0.99772
10     52.631579    -4.460488    1.079040    1.00000
11     57.894737    -5.954716    1.121081    1.00000
12     63.157895    -7.254722    1.159093    1.00000
13     68.421053    -8.466345    1.202499    1.00000
14     73.684211    -9.870535    1.249719    1.00000
15     78.947368   -11.240783    1.292764    1.00000
16     84.210526   -12.465459    1.336721    1.00000
17     89.473684   -13.611748    1.360644    1

In [5]:
summary_5mm = pd.DataFrame({
                              'Time (years)': result['times'],
                              'Sample mean':  samples.mean(axis=0),
                              'Sample std':   samples.std(axis=0),
                              'P(g <= 5)':    (samples <= 5.0).mean(axis=0),
                           })
summary_5mm

    Time (years)  Sample mean  Sample std  P(g <= 5)
0       0.000000    25.572298    0.498422    0.00000
1       5.263158    21.546289    0.548102    0.00000
2      10.526316    17.199912    0.611805    0.00000
3      15.789474    12.570653    0.663926    0.00000
4      21.052632     8.069692    0.751471    0.00000
5      26.315789     4.805568    0.846623    0.58748
6      31.578947     2.747629    0.901974    0.99392
7      36.842105     0.829488    0.946717    1.00000
8      42.105263    -1.050128    0.986104    1.00000
9      47.368421    -2.877336    1.030354    1.00000
10     52.631579    -4.460488    1.079040    1.00000
11     57.894737    -5.954716    1.121081    1.00000
12     63.157895    -7.254722    1.159093    1.00000
13     68.421053    -8.466345    1.202499    1.00000
14     73.684211    -9.870535    1.249719    1.00000
15     78.947368   -11.240783    1.292764    1.00000
16     84.210526   -12.465459    1.336721    1.00000
17     89.473684   -13.611748    1.360644    1